Group 7 Final Project - DATASCI 151


Student ID Numbers: Alec Anghel (2697818), Stanley Yoo (), Elsa Mathew-Lewis (2677374), Kimberly Yang ()

# Formula One Data Analysis

### Introduction
Formula One (F1) is a competitive motorsport, where both driver skill and car performance play an important role in determing race outcomes.  As a result, understanding what factors contribute to success in Formula One is an important and interesting data analysis problem as it can be used to better understand things such as: what drives a winning team, what makes a team better, and what can a team do to improve their outcomes. Through this project we aim to address some of these questions. 

One of these investigations is the relationship between qualifying performance and race results to determine whether starting position is a strong indicator of success. Specifically, we analyze whether drivers who perform well in qualifying sessions tend to achieve better final positions and earn more points in races. Using multiple datasets from F1 race records, we merge and analyze qualifying and race results to explore this relationship. We also examine the data to determine if a driver's nationality correlates with a faster lap time and at what age F1 drivers typically reach their peak performances. Our findings suggest that there is a strong association between qualifying position and race outcomes, although other factors such as race conditions and strategy may also influence final performance. Through our exploration of driver demographics, race dates, and over 500,000 individual lap times, it became evident that Canadian and Italian drivers hold the fastest historical average pace. Additionally, our analysis shows a clear aging curve, showing drivers reach their peak speed between the ages of 26 and 32. The project is structured by first introducing, describing, and cleaning the datasets, and then followed with the methods of analysis, and the results of our findings.

### Data Description
The data for this project came from a Formula 1 dataset found on Kaggle. Five main data sets were used for this analysis.
"drivers.csv": A data set with 854 containing observations of driver demographics. It cotains their names, dates of brith, and nationalities.
"lap_times.csv": A data set that contains 528,785 observations of individual lap times, race IDs, and milliseconds recorded.
"races.csv": A data set that contains 9,395 race dates. We will use this information alongside race ID to find the drivers exact age at the time of a race.
"qualifying.csv": A data set containing 25,660 observations about each driver's starting information for a race such as position and the driver identification
"results.csv": A data set containing 1,079 observations relating to final race outcomes, including finishing position and points earned.

In [2]:
# importing packages
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [5]:
# loading the datasets
drivers = pd.read_csv("f1_data/drivers.csv")
lap_times = pd.read_csv("f1_data/lap_times.csv")
qualifying = pd.read_csv("f1_data/qualifying.csv")
results = pd.read_csv("f1_data/results.csv")
races = pd.read_csv("f1_data/races.csv")

### Data Cleaning
Before we merge our data, the raw data needs to be cleaned for more accurate statiscal computing. In the drivers table, we replaced missing values ("\N") with "NaN" values, droped the unnecessary URL column, and converted the `dob` (Date of Birth) column into a usable datetime format. In the lap times table, we created a new column to convert the  "milliseconds" into  "seconds" to make our final plots and tables more understandable.

In [ ]:
# First thing to do is to clean the data sets
# Let's first clean the drivers data set

# Replace the \N in the numbers with the NaN value
drivers = drivers.replace(r'\N', np.nan)

# We don't really need the url column so we can drop that
drivers = drivers.drop(columns=['url'])

# Convert the date of birth to an actual date of birth format
drivers['dob'] = pd.to_datetime(drivers['dob'])

# Let's convert the number column into actual numeric values
drivers['number'] = pd.to_numeric(drivers['number'])

display(drivers)

In [ ]:
# Ok now lets clean the lap_times data set
# It already looks pretty clean, only thing I want to do is convert miliseconds into seconds
lap_times['seconds'] = lap_times['milliseconds'] / 1000
display(lap_times)

### Merging Procedure
To analyze performance by driver attributes, we have to combine our tables. We performed an inner join on the "lap_times" and "drivers" datasets using the "driverId" column, since it appeared in both datasets. An inner join makes it so that our combined dataset only keeps rows where we have both a valid lap time and the corresponding driver's demographic details.

In [ ]:
# Now lets merge the data on the driverID column
# I'm going to use an inner join so that we only keep the rows where we have both the driver's details and a valid lap time
f1_data = pd.merge(lap_times, drivers, on='driverId', how='inner')

display(f1_data)

To compare qualifying performance with race outcomes, the qualifying and results datasets were merged together. These two tables share the variables `raceId` and `driverId`, which uniquely identify a driver in a specific race. An inner merge was used so that only observations appearing in both datasets were kept. This allows each row in the merged table to represent one driver in one race, with both that driver’s qualifying position and final race result included in the same dataset. After merging, the relevant columns were renamed for clarity so that qualifying position and race finishing position could be distinguished more easily in the analysis.

After merging the datasets, several data cleaning steps were performed to ensure the dataset was suitable for analysis. First, only the relevant columns needed for this study were retained, including qualifying position, race finishing position, and points. Rows containing missing values were removed to avoid inaccuracies in the analysis. Additionally, the dataset was checked for consistency in data types, and the columns were kept in numerical form to allow for proper comparisons and calculations. These cleaning steps ensure that the dataset is accurate and ready for analyzing the relationship between qualifying performance and race outcomes.

In [ ]:
# mergining qualifying and results data sets

qualifying_small = qualifying[["raceId", "driverId", "position"]]
results_small = results[["raceId", "driverId", "positionOrder", "points"]]

merged = pd.merge(qualifying_small, results_small, on=["raceId", "driverId"], how="inner")

merged = merged.rename(columns={
    "position": "qualifying_position",
    "positionOrder": "race_position"
})

merged.head()

# Drop missing values
merged = merged.dropna()

# Ensure numeric data types
merged["qualifying_position"] = pd.to_numeric(merged["qualifying_position"])
merged["race_position"] = pd.to_numeric(merged["race_position"])
merged["points"] = pd.to_numeric(merged["points"])

# Reset index after cleaning
merged = merged.reset_index(drop=True)

# Check cleaned data
merged.head()
merged.shape

### Main Columns and Descriptive Statistics
Our main columns of interest for the analysis are "nationality", "seconds", and later, the driver's calculated "age". Below is a table of descriptive statistics for our seconds variable.

The main columns used in our secondary analysis are qualifying_position, race_position, and points. The qualifying_position column represents the position a driver starts in after the qualifying session, while the race_position column represents the driver’s final finishing position in the race. The points column indicates how many championship points the driver earned in that race. These columns are important because they allow us to directly compare starting performance with race outcomes. By examining these variables, we can analyze whether drivers who start in better positions tend to finish in better positions and earn more points.

In [ ]:
# Compute descriptive statistics for our numerical column and round to 2 decimal places
descriptive_stats = f1_data[['seconds']].describe().round(2)
display(descriptive_stats)

In [ ]:
# summary statisitcs of relevant columns, rounded to two decimal places
display(merged[["qualifying_position", "race_position", "points"]].describe().round(2))